# v4.3-lite — training (3 variants)

Drives one training run. Pick a **variant** in section 1 by setting `CONFIG['variant']`:

| variant | top-level head | per-SF cap | what it tests |
|---|---|---|---|
| `three_class_balanced` | DNA / LTR / LINE | 100 ≤ N ≤ 3000 | direct lite vs v4.3 baseline |
| `three_class_unbalanced` | DNA / LTR / LINE | none | does manual SF subsampling matter? |
| `binary_dna` | DNA vs non-DNA | 100 ≤ N ≤ 3000 | apples-to-apples vs original binary v4 overfit gap |
| `binary_dna_natural` | DNA vs non-DNA | no SF cap | preserves natural ~8% / 92% prior so train prior matches eval prior |

Outputs go to `results/<variant>/` next to this notebook. Run the notebook three times (one per variant) — each run is independent. **The slow cells are clearly marked**; for a fast smoke check, set `CONFIG['smoke'] = True` in section 1.

All shared model/dataset code lives in `_lib.py` (mirrors notebook 01).

## 1. Configuration

Edit only this cell between runs.

In [1]:
CONFIG = {
    # ---- variant selector ----
    'variant': 'binary_dna',   # 'three_class_balanced' | 'three_class_unbalanced' | 'binary_dna' | 'binary_dna_natural'
    'smoke':   False,                     # if True: subsample data + epochs for a quick sanity check

    # ---- data ----
    'fasta_path':  '../../../data/vgp/all_vgp_tes.fa',
    'label_path':  '../../../data/vgp/20260120_features_sf',
    'exclude_genomes': {'mOrnAna', 'bTaeGut', 'rAllMis'},   # benchmark hold-out (matches v4.3)
    'min_class_count': 100,
    'random_state': 42,

    # ---- training ----
    'batch_size':    16,
    'epochs':        40,
    'lr':            3e-4,
    'weight_decay':  5e-4,                # v4.3: 1e-4 -> 5e-4 (lever B)
    'grad_clip':     1.0,
    'label_smoothing': 0.1,
    'class_weight':  1.0,                 # weight on top-level loss
    'sf_weight':     1.0,                 # weight on superfamily loss

    # ---- regularisation knobs (lever B) ----
    'augment':  True,                     # RC flip + N-noise
    'mixup_alpha': 0.0,                   # 0.0 disables embedding-mixup on top head

    # ---- CV / early stopping ----
    'n_folds':           5,
    'test_size':         0.2,
    'patience':          7,               # epochs without val-SF-F1 improvement
    'es_metric':         'val_sf_f1',     # **single-head** criterion (was combined in v4.3)
    'topk_checkpoints':  3,
    'save_dir_root':     './results',
}

# ---- variant-specific overrides ----
if CONFIG['variant'] == 'three_class_balanced':
    CONFIG['keep_classes'] = ('DNA', 'LTR', 'LINE')
    CONFIG['max_per_sf']   = 3000
    CONFIG['binary_dna']   = False
elif CONFIG['variant'] == 'three_class_unbalanced':
    CONFIG['keep_classes'] = ('DNA', 'LTR', 'LINE')
    CONFIG['max_per_sf']   = None         # NO cap (lever C)
    CONFIG['binary_dna']   = False
elif CONFIG['variant'] == 'binary_dna':
    CONFIG['keep_classes'] = ('DNA', 'LTR', 'LINE')   # SF table identical; top label collapsed below
    CONFIG['max_per_sf']   = 3000
    CONFIG['binary_dna']   = True
elif CONFIG['variant'] == 'binary_dna_natural':
    # Natural-prior binary: keep all top-level classes, no SF cap, so the
    # train prior of DNA vs non-DNA matches the natural ~8/92 eval prior.
    CONFIG['keep_classes'] = None         # None -> keep every top-level class that passes min_class_count
    CONFIG['max_per_sf']   = None         # NO cap; preserves natural class imbalance
    CONFIG['binary_dna']   = True
else:
    raise ValueError(f"unknown variant {CONFIG['variant']!r}")

# ---- smoke-mode overrides ----
if CONFIG['smoke']:
    CONFIG['epochs']      = 2
    CONFIG['n_folds']     = 2
    CONFIG['patience']    = 99            # never trigger
    CONFIG['_smoke_n']    = 800           # cap trainval samples
    CONFIG['save_dir_root'] = './results_smoke'


from pathlib import Pathprint(f"Save dir: {SAVE_DIR.resolve()}")

SAVE_DIR = Path(CONFIG['save_dir_root']) / CONFIG['variant']print(f"Smoke:    {CONFIG['smoke']}")

SAVE_DIR.mkdir(parents=True, exist_ok=True)print(f"Variant:  {CONFIG['variant']}")

SyntaxError: invalid syntax (3309432390.py, line 67)

## 2. Imports & device

In [ ]:
import gc
import json
import math
import time
import heapq
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

import _lib as L

torch.manual_seed(CONFIG['random_state'])
np.random.seed(CONFIG['random_state'])
DEVICE = L.resolve_device()
print(f'PyTorch {torch.__version__} on {DEVICE}')

## 3. Load FASTA + labels  *(slow cell — reads ~all_vgp_tes.fa)*

In [ ]:
t0 = time.time()
headers, sequences = L.read_fasta(CONFIG['fasta_path'])
label_dict, class_dict, top_class_to_id = L.load_labels(CONFIG['label_path'], keep_classes=CONFIG['keep_classes'])
print(f'Loaded {len(headers):,} FASTA records and {len(label_dict):,} labelled headers in {time.time()-t0:.1f}s')

## 4. Filter, exclude benchmark genomes, optional per-SF cap

In [ ]:
prep = L.filter_and_subsample(
    headers, sequences, label_dict, class_dict,
    exclude_genomes=CONFIG['exclude_genomes'],
    min_class_count=CONFIG['min_class_count'],
    max_per_sf=CONFIG['max_per_sf'],
    random_state=CONFIG['random_state'],
)
del headers, sequences
gc.collect()

all_h        = prep['headers']
all_s        = prep['sequences']
all_top      = prep['toplevel']
all_sf       = prep['sf']
all_tags     = prep['tags']
sf_names     = prep['sf_names']
sf_to_id     = prep['sf_to_id']
n_sf         = len(sf_names)
class_names  = list(top_class_to_id.keys())

# binary_dna: collapse top-level to {DNA -> 1, non-DNA -> 0}
if CONFIG['binary_dna']:
    dna_id = class_names.index('DNA')
    all_top_used = (all_top == dna_id).astype(np.int64)
    toplevel_names = ['non-DNA', 'DNA']
else:
    all_top_used = all_top.astype(np.int64)
    toplevel_names = class_names
n_top = len(toplevel_names)

print(f'Excluded {prep["n_excluded_genomes"]:,} sequences from benchmark genomes')
print(f'After filtering: {len(all_h):,} sequences across {n_sf} superfamilies')
print(f'Top-level head ({n_top}-way):')
for i, name in enumerate(toplevel_names):
    print(f'  {name}: {(all_top_used == i).sum():,}')

if CONFIG['smoke']:
    rng = np.random.default_rng(CONFIG['random_state'])
    keep = rng.choice(len(all_h), size=min(CONFIG['_smoke_n'], len(all_h)), replace=False)
    keep = sorted(keep.tolist())
    all_h = [all_h[i] for i in keep]
    all_s = [all_s[i] for i in keep]
    all_top_used = all_top_used[keep]
    all_sf = all_sf[keep]
    all_tags = [all_tags[i] for i in keep]
    print(f'\nSMOKE mode: subsampled to {len(all_h)} sequences')

## 5. Pre-compute k-mer features  *(slow cell — pure-Python loop, scales with dataset size)*

Result is reused across all 5 folds and the final test eval.

In [ ]:
feat = L.KmerWindowFeaturizer(
    k=L.KMER_K, dim=L.KMER_DIM, window=L.KMER_WINDOW, stride=L.KMER_STRIDE,
)
t0 = time.time()
all_kmer = [feat.featurize_sequence(s) for s in tqdm(all_s, desc='Featurizing')]
print(f'Featurized {len(all_kmer):,} sequences in {time.time()-t0:.1f}s')
print(f'  feature dim: {all_kmer[0].shape[-1]} (= KMER_DIM + 1 position)')

## 6. Train / test split + 5-fold CV indices

Stratified on **superfamily** (not top-level class) — finer-grained stratification reduces SF train/val leakage.

In [ ]:
strat = np.asarray([sf_to_id[t] for t in all_tags])
idx_trainval, idx_test = train_test_split(
    np.arange(len(all_h)),
    test_size=CONFIG['test_size'],
    stratify=strat,
    random_state=CONFIG['random_state'],
)

trainval = {
    'h':    [all_h[i]    for i in idx_trainval],
    's':    [all_s[i]    for i in idx_trainval],
    'top':  all_top_used[idx_trainval],
    'sf':   all_sf[idx_trainval],
    'k':    [all_kmer[i] for i in idx_trainval],
    'strat': strat[idx_trainval],
}
test = {
    'h':    [all_h[i]    for i in idx_test],
    's':    [all_s[i]    for i in idx_test],
    'top':  all_top_used[idx_test],
    'sf':   all_sf[idx_test],
    'k':    [all_kmer[i] for i in idx_test],
}
skf = StratifiedKFold(n_splits=CONFIG['n_folds'], shuffle=True, random_state=CONFIG['random_state'])
fold_splits = list(skf.split(trainval['h'], trainval['strat']))

print(f'Trainval: {len(trainval["h"]):,}    Held-out test: {len(test["h"]):,}')
print(f'{CONFIG["n_folds"]}-fold CV indices ready (rotating per epoch).')

# Free the full lists; we keep only the trainval/test partitions.
del all_h, all_s, all_top_used, all_sf, all_kmer, all_tags
gc.collect()

## 7. Build model, optimiser, losses

- AdamW lr=1e-3, weight_decay=5e-4 (v4.3 used 1e-4).
- Cosine LR schedule.
- Label-smoothed CE with inverse-sqrt class weights for both heads.
- Gradient clipping max-norm 1.0.
- Embedding MixUp (lam ~ Beta(α, α)) on the top-level head only when `mixup_alpha > 0`.

In [ ]:
model = L.HybridTEClassifierV43Lite(
    num_toplevel=n_top,
    num_superfamilies=n_sf,
).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {n_params:,} ({n_params/1e6:.3f} M)')

top_w = L.compute_class_weights(trainval['top'], n_top, mode='inv_sqrt')
sf_w  = L.compute_class_weights(trainval['sf'],  n_sf,  mode='inv_sqrt')
top_w_t = torch.tensor(top_w, dtype=torch.float32, device=DEVICE)
sf_w_t  = torch.tensor(sf_w,  dtype=torch.float32, device=DEVICE)

loss_top = nn.CrossEntropyLoss(label_smoothing=CONFIG['label_smoothing'], weight=top_w_t).to(DEVICE)
loss_sf  = nn.CrossEntropyLoss(label_smoothing=CONFIG['label_smoothing'], weight=sf_w_t).to(DEVICE)

opt = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scheduler = CosineAnnealingLR(opt, T_max=CONFIG['epochs'], eta_min=CONFIG['lr'] * 0.01)

print('top-level class weights:', dict(zip(toplevel_names, [round(float(x),3) for x in top_w])))
print(f'sf weights: min={sf_w.min():.3f}  max={sf_w.max():.3f}  mean={sf_w.mean():.3f}')

## 8. Top-K checkpoint manager (in-memory + disk)

In [ ]:
class TopKCheckpoint:
    """Keep the K best checkpoints by score (higher is better). Replaces older worse ones on disk."""
    def __init__(self, save_dir, prefix='hybrid_v4_3_lite', k=3):
        self.dir = Path(save_dir); self.dir.mkdir(parents=True, exist_ok=True)
        self.prefix = prefix
        self.k = k
        self.heap = []          # (score, epoch)
        self.paths = {}         # epoch -> Path

    def maybe_save(self, score, epoch, model, meta):
        if len(self.heap) < self.k:
            self._save(score, epoch, model, meta); heapq.heappush(self.heap, (score, epoch))
            return True
        if score > self.heap[0][0]:
            old_score, old_ep = heapq.heappop(self.heap)
            p = self.paths.pop(old_ep, None)
            if p and p.exists(): p.unlink()
            self._save(score, epoch, model, meta); heapq.heappush(self.heap, (score, epoch))
            return True
        return False

    def _save(self, score, epoch, model, meta):
        sd = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        path = self.dir / f'{self.prefix}_epoch{epoch}.pt'
        torch.save({'state_dict': sd, 'epoch': epoch, 'score': score, **meta}, path)
        self.paths[epoch] = path

    def best(self):
        if not self.heap: return None
        score, epoch = max(self.heap)
        return torch.load(self.paths[epoch], weights_only=False, map_location='cpu')

ckpt_mgr = TopKCheckpoint(SAVE_DIR, prefix=f"hybrid_v43_lite_{CONFIG['variant']}",
                          k=CONFIG['topk_checkpoints'])

## 9. Train / eval helpers

In [ ]:
def make_loader(part_idx, split: str):
    """Build a DataLoader from indices into the trainval dict, or use the test dict if split=='test'."""
    if split == 'test':
        d = test
        ds = L.HybridDataset(d['h'], d['s'], d['top'], d['sf'], d['k'],
                              augment=L.AugmentConfig(enabled=False))
        shuffle = False
    else:
        d = trainval
        sub = lambda lst: [lst[i] for i in part_idx]
        ds = L.HybridDataset(
            sub(d['h']), sub(d['s']), d['top'][part_idx], d['sf'][part_idx], sub(d['k']),
            augment=L.AugmentConfig(enabled=(split == 'train' and CONFIG['augment'])),
            rng_seed=CONFIG['random_state'] if split == 'val' else None,
        )
        shuffle = (split == 'train')
    return DataLoader(ds, batch_size=CONFIG['batch_size'], shuffle=shuffle,
                      num_workers=0, collate_fn=L.collate_hybrid)


def _to_dev(*tensors):
    return [t.to(DEVICE, non_blocking=True) for t in tensors]


def train_one_epoch(loader):
    model.train()
    tot, tot_top, tot_sf, n = 0.0, 0.0, 0.0, 0
    for _, X, mask, Y_top, Y_sf, x_g, ei, bv in loader:
        X, mask, Y_top, Y_sf, x_g, ei, bv = _to_dev(X, mask, Y_top, Y_sf, x_g, ei, bv)

        # MixUp on the top-level head (sample lam ~ Beta(alpha, alpha))
        if CONFIG['mixup_alpha'] > 0:
            lam = float(np.random.beta(CONFIG['mixup_alpha'], CONFIG['mixup_alpha']))
            perm = torch.randperm(X.size(0), device=DEVICE)
            top_l, sf_l, _ = model(X, mask, x_g, ei, bv, mixup_lam=lam, mixup_perm=perm)
            top_l = top_l.clamp(-30.0, 30.0); sf_l = sf_l.clamp(-30.0, 30.0)
            l_top = lam * loss_top(top_l, Y_top) + (1 - lam) * loss_top(top_l, Y_top[perm])
        else:
            top_l, sf_l, _ = model(X, mask, x_g, ei, bv)
            top_l = top_l.clamp(-30.0, 30.0); sf_l = sf_l.clamp(-30.0, 30.0)
            l_top = loss_top(top_l, Y_top)
        l_sf = loss_sf(sf_l, Y_sf)
        loss = CONFIG['class_weight'] * l_top + CONFIG['sf_weight'] * l_sf

        if not torch.isfinite(loss):
            print(f'  [WARN] non-finite loss at ep={ep} step (skipped)')
            opt.zero_grad(); continue

        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['grad_clip'])
        opt.step()

        b = X.size(0)
        tot += loss.item() * b; tot_top += l_top.item() * b; tot_sf += l_sf.item() * b; n += b
    return tot / n, tot_top / n, tot_sf / n


@torch.no_grad()
def evaluate(loader):
    """Returns (top_acc, top_f1, sf_acc, sf_f1, gate_cnn_mean, gate_gnn_mean, top_pred, top_true, sf_pred, sf_true)."""
    model.eval()
    tp, tt, sp, st, gc_, gg = [], [], [], [], [], []
    for _, X, mask, Y_top, Y_sf, x_g, ei, bv in loader:
        X, mask, x_g, ei, bv = _to_dev(X, mask, x_g, ei, bv)
        top_l, sf_l, gates = model(X, mask, x_g, ei, bv)
        tp.append(top_l.argmax(1).cpu().numpy()); tt.append(Y_top.numpy())
        sp.append(sf_l.argmax(1).cpu().numpy());  st.append(Y_sf.numpy())
        gc_.append(gates[:, 0].cpu().numpy());     gg.append(gates[:, 1].cpu().numpy())
    tp, tt, sp, st = (np.concatenate(x) for x in (tp, tt, sp, st))
    gc_, gg = np.concatenate(gc_), np.concatenate(gg)
    return (
        accuracy_score(tt, tp),
        f1_score(tt, tp, average='macro', zero_division=0),
        accuracy_score(st, sp),
        f1_score(st, sp, average='macro', zero_division=0),
        float(gc_.mean()), float(gg.mean()),
        tp, tt, sp, st,
    )

## 10. Training loop  *(this is the long-running cell)*

Per epoch:
1. Pick the rotating fold (epoch mod n_folds).
2. Train one pass with augmentations + MixUp.
3. Evaluate on the fold's validation set + the **train fold** (no aug) so we can track the train→val SF-F1 gap directly.
4. Step cosine scheduler. Save checkpoint if in top-K. Early stop on `val_sf_f1` patience.

In [ ]:
history = {
    'epoch': [], 'fold': [], 'lr': [],
    'train_loss': [], 'train_top_loss': [], 'train_sf_loss': [],
    'train_top_f1': [], 'train_sf_f1': [],
    'val_top_acc': [], 'val_top_f1': [], 'val_sf_acc': [], 'val_sf_f1': [],
    'gate_cnn': [], 'gate_gnn': [],
}
best_es = -math.inf
bad_streak = 0
t0 = time.time()

for ep in range(1, CONFIG['epochs'] + 1):
    fi = (ep - 1) % CONFIG['n_folds']
    tr_idx, va_idx = fold_splits[fi]
    train_loader = make_loader(tr_idx, 'train')
    val_loader   = make_loader(va_idx, 'val')
    train_eval_loader = make_loader(tr_idx, 'val')   # no aug, for the gap measurement

    tr_loss, tr_top_l, tr_sf_l = train_one_epoch(train_loader)
    scheduler.step()

    _, tr_top_f1, _, tr_sf_f1, _, _, *_ = evaluate(train_eval_loader)
    va_top_acc, va_top_f1, va_sf_acc, va_sf_f1, gate_c, gate_g, *_ = evaluate(val_loader)

    history['epoch'].append(ep);            history['fold'].append(fi + 1)
    history['lr'].append(opt.param_groups[0]['lr'])
    history['train_loss'].append(tr_loss);  history['train_top_loss'].append(tr_top_l); history['train_sf_loss'].append(tr_sf_l)
    history['train_top_f1'].append(tr_top_f1); history['train_sf_f1'].append(tr_sf_f1)
    history['val_top_acc'].append(va_top_acc); history['val_top_f1'].append(va_top_f1)
    history['val_sf_acc'].append(va_sf_acc);   history['val_sf_f1'].append(va_sf_f1)
    history['gate_cnn'].append(gate_c);    history['gate_gnn'].append(gate_g)

    sf_gap = tr_sf_f1 - va_sf_f1
    es_score = va_sf_f1 if CONFIG['es_metric'] == 'val_sf_f1' else 0.5 * (va_top_f1 + va_sf_f1)

    print(
        f'Ep {ep:3d} fold{fi+1} | loss {tr_loss:.4f} | '
        f'TOP train {tr_top_f1:.3f} val {va_top_f1:.3f}  | '
        f'SF  train {tr_sf_f1:.3f} val {va_sf_f1:.3f}  gap {sf_gap:+.3f} | '
        f'gate C/G {gate_c:.2f}/{gate_g:.2f}'
    )

    saved = ckpt_mgr.maybe_save(
        score=es_score, epoch=ep, model=model,
        meta={'config': {k: (list(v) if isinstance(v, set) else v) for k, v in CONFIG.items()},
              'sf_names': sf_names, 'toplevel_names': toplevel_names,
              'history_so_far': dict(history)},
    )
    if saved:
        print(f'  saved (top-{CONFIG["topk_checkpoints"]}: {sorted([e for _, e in ckpt_mgr.heap])})')

    if es_score > best_es:
        best_es = es_score; bad_streak = 0
    else:
        bad_streak += 1
        if bad_streak >= CONFIG['patience']:
            print(f'\nEarly stop: no improvement in {CONFIG["es_metric"]} for {CONFIG["patience"]} epochs.')
            break

wall = time.time() - t0
print(f'\nTraining done in {wall/60:.1f} min ({wall/len(history["epoch"]):.1f}s/epoch). '
      f'Best {CONFIG["es_metric"]} = {best_es:.4f}')

## 11. Final evaluation on the held-out VGP test set

Loads the best top-K checkpoint by validation score, re-runs on the held-out test split, and prints the train→test SF-F1 gap that we are trying to shrink.

In [ ]:
best_ckpt = ckpt_mgr.best()
model.load_state_dict(best_ckpt['state_dict'])
best_epoch = best_ckpt['epoch']
print(f'Loaded best checkpoint from epoch {best_epoch} (val score {best_ckpt["score"]:.4f})')

test_loader = make_loader(None, 'test')
te_top_acc, te_top_f1, te_sf_acc, te_sf_f1, te_gc, te_gg, te_top_pred, te_top_true, te_sf_pred, te_sf_true = evaluate(test_loader)

tr_top_f1_best = history['train_top_f1'][best_epoch - 1]
tr_sf_f1_best  = history['train_sf_f1'][best_epoch - 1]
va_top_f1_best = history['val_top_f1'][best_epoch - 1]
va_sf_f1_best  = history['val_sf_f1'][best_epoch - 1]

print('\n--- HELD-OUT VGP TEST ---')
print(f'  TOP   acc {te_top_acc:.4f}   macro-F1 {te_top_f1:.4f}')
print(f'  SF    acc {te_sf_acc:.4f}   macro-F1 {te_sf_f1:.4f}')
print(f'  Gate  CNN {te_gc:.2f}   GNN {te_gg:.2f}')
print('\n--- TRAIN -> VAL -> TEST F1 GAPS (best epoch) ---')
print(f'  TOP-F1   train {tr_top_f1_best:.4f}  val {va_top_f1_best:.4f}  test {te_top_f1:.4f}   '
      f'gap(tr-te) {tr_top_f1_best - te_top_f1:+.4f}')
print(f'  SF-F1    train {tr_sf_f1_best:.4f}  val {va_sf_f1_best:.4f}  test {te_sf_f1:.4f}   '
      f'gap(tr-te) {tr_sf_f1_best - te_sf_f1:+.4f}')

print('\n--- TOP-LEVEL classification report ---')
print(classification_report(te_top_true, te_top_pred, target_names=toplevel_names, digits=3, zero_division=0))

## 12. Save results

Two files dropped into the variant folder for `03_eval_and_compare.ipynb` to consume:
- `results.pt` — full history + test preds + the per-sample gate weights;
- `test_metrics.json` — small summary table (also human-readable).

In [ ]:
results_payload = {
    'variant':         CONFIG['variant'],
    'config':          {k: (list(v) if isinstance(v, set) else v) for k, v in CONFIG.items()},
    'best_epoch':      best_epoch,
    'history':         history,
    'test_top_pred':   te_top_pred.tolist(),
    'test_top_true':   te_top_true.tolist(),
    'test_sf_pred':    te_sf_pred.tolist(),
    'test_sf_true':    te_sf_true.tolist(),
    'toplevel_names':  toplevel_names,
    'sf_names':        sf_names,
    'sf_to_id':        sf_to_id,
    'n_params':        n_params,
}
torch.save(results_payload, SAVE_DIR / 'results.pt')

summary = {
    'variant': CONFIG['variant'],
    'n_params': n_params,
    'best_epoch': best_epoch,
    'epochs_run': len(history['epoch']),
    'train_top_f1': tr_top_f1_best,
    'val_top_f1':   va_top_f1_best,
    'test_top_acc': te_top_acc, 'test_top_f1': te_top_f1,
    'train_sf_f1':  tr_sf_f1_best,
    'val_sf_f1':    va_sf_f1_best,
    'test_sf_acc':  te_sf_acc,  'test_sf_f1':  te_sf_f1,
    'top_f1_gap_train_test': tr_top_f1_best - te_top_f1,
    'sf_f1_gap_train_test':  tr_sf_f1_best  - te_sf_f1,
    'gate_cnn_test': te_gc, 'gate_gnn_test': te_gg,
}
(SAVE_DIR / 'test_metrics.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
print(f'\nSaved -> {SAVE_DIR.resolve()}')